In [1]:
import numpy as np
import pandas as pd

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Load modelling-ready dataset
df_model = pd.read_csv("../data/processed/df_young_model_ready_2024.csv")

# Define target + protected attrs
y = df_model["high_risk"].astype(int)

A = df_model[["sex_of_driver", "age_band_of_driver"]].copy()

# Features (exclude target + protected)
X = df_model.drop(columns=["high_risk", "sex_of_driver", "age_band_of_driver"])

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test, A_train, A_test = train_test_split(
    X, y, A,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("y_test distribution:", np.bincount(y_test))

Train: (10700, 54) Test: (2676, 54)
y_test distribution: [1967  709]


## A) Logistic Regression (scaled + class-weight balanced)

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lr_bal = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE))
])

lr_bal.fit(X_train, y_train)
y_proba_lr = lr_bal.predict_proba(X_test)[:, 1]

## B) Random Forest (baseline model we used in notebook 04)

In [3]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train, y_train)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

## C) XGBoost (same as the baseline settings in notebook 04)

In [4]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

In [5]:
THRESHOLDS = {
    "LR_balanced": 0.5,
    "RandomForest": 0.3,
    "XGBoost": 0.3
}

probas = {
    "LR_balanced": y_proba_lr,
    "RandomForest": y_proba_rf,
    "XGBoost": y_proba_xgb
}

preds = {name: (p >= THRESHOLDS[name]).astype(int) for name, p in probas.items()}

In [6]:
from fairlearn.metrics import MetricFrame, selection_rate, demographic_parity_difference, equalized_odds_difference
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## Build a clean metrics dictionary

We’ll compute:
- overall performance
- group performance by sex_of_driver and by age_band_of_driver
- fairness gaps:
    - Demographic parity difference
    - Equalized odds difference
    - selection rate by group

In [7]:
metrics = {
    "accuracy": accuracy_score,
    "precision": lambda yt, yp: precision_score(yt, yp, zero_division=0),
    "recall": recall_score,
    "f1": lambda yt, yp: f1_score(yt, yp, zero_division=0),
    "selection_rate": selection_rate
}

### A) Fairness by sex

In [8]:
def fairness_report(model_name, y_true, y_pred, sensitive_series, sensitive_name):
    mf = MetricFrame(
        metrics=metrics,
        y_true=y_true,
        y_pred=y_pred,
        sensitive_features=sensitive_series
    )
    print(f"\n=== {model_name} | Sensitive attribute: {sensitive_name} ===")
    print("Overall:")
    print(mf.overall)
    print("\nBy group:")
    print(mf.by_group)

    # Fairness gaps
    dp_diff = demographic_parity_difference(y_true, y_pred, sensitive_features=sensitive_series)
    eo_diff = equalized_odds_difference(y_true, y_pred, sensitive_features=sensitive_series)

    print(f"\nDemographic parity difference: {dp_diff:.4f}")
    print(f"Equalized odds difference:     {eo_diff:.4f}")

for name, y_pred in preds.items():
    fairness_report(
        model_name=name,
        y_true=y_test,
        y_pred=y_pred,
        sensitive_series=A_test["sex_of_driver"],
        sensitive_name="sex_of_driver"
    )


=== LR_balanced | Sensitive attribute: sex_of_driver ===
Overall:
accuracy          0.593423
precision         0.351139
recall            0.630465
f1                0.451060
selection_rate    0.475710
dtype: float64

By group:
               accuracy  precision    recall        f1  selection_rate
sex_of_driver                                                         
1              0.583864   0.374194  0.632727  0.470270        0.493631
2              0.616162   0.288630  0.622642  0.394422        0.433081

Demographic parity difference: 0.0605
Equalized odds difference:     0.0508

=== RandomForest | Sensitive attribute: sex_of_driver ===
Overall:
accuracy          0.659567
precision         0.406827
recall            0.622003
f1                0.491913
selection_rate    0.405082
dtype: float64

By group:
               accuracy  precision    recall        f1  selection_rate
sex_of_driver                                                         
1              0.641189   0.425178  0.65

### B) Fairness by age band

In [9]:
for name, y_pred in preds.items():
    fairness_report(
        model_name=name,
        y_true=y_test,
        y_pred=y_pred,
        sensitive_series=A_test["age_band_of_driver"],
        sensitive_name="age_band_of_driver"
    )


=== LR_balanced | Sensitive attribute: age_band_of_driver ===
Overall:
accuracy          0.593423
precision         0.351139
recall            0.630465
f1                0.451060
selection_rate    0.475710
dtype: float64

By group:
                    accuracy  precision    recall        f1  selection_rate
age_band_of_driver                                                         
4                   0.580017   0.356358  0.725240  0.477895        0.539373
5                   0.604013   0.345912  0.555556  0.426357        0.425418

Demographic parity difference: 0.1140
Equalized odds difference:     0.1697

=== RandomForest | Sensitive attribute: age_band_of_driver ===
Overall:
accuracy          0.659567
precision         0.406827
recall            0.622003
f1                0.491913
selection_rate    0.405082
dtype: float64

By group:
                    accuracy  precision    recall        f1  selection_rate
age_band_of_driver                                                         
